第10回講義
========

コンピュータ上のデータには、映像や音声といった人間の五感に直接作用するデータ(マルチメディアデータ)もあります。いずれのデータについても、A/D変換(量子化)といった方法で現実の世界の情報を最終的にデジタルデータ(数値の羅列)で表す必要があります。そのデータを再び人間が理解できる情報とするには、ディスプレイやスピーカーと言ったデバイスに出力する方法も必要です。

一度デジタル化されたマルチメディアデータに対して、数学的な演算を行うことができるので、そのデータを加工したり、データそのものを分析対象とすることができます。映像データを分析することによって画像解析や映像認識・顔認識、音声データを分析したり再合成することによって音声認識や音声合成といった技術に応用することができます。

画像データの可視化
--------------

### 画像データの入出力と解析

画像データを入出力するためのモジュールとしてその目的によって様々なものがありますが、ここでは、**OpenCV**(Open Source Computer Vision Library)を取り上げます。OpenCVとはインテルが開発・公開したオープンソースのコンピュータビジョン向けライブラリです。

OpenCVは非常に多機能なモジュールで、ほとんど自分で何かを作成する必要はないのですが、まずは画像の読み込みだけの機能を使用して、画像データの取り扱い方法について学びます。

### OpenCVのインストール

In [ ]:
!pip install --user opencv-python-headless

インストール後はカーネルを再起動する。

メニューから: 「カーネル」 > 「カーネルを再起動...」 > 「再起動」ボタンをクリック

<mark>練習1</mark> OpenCVモジュールを用いて、`chapel.jpg`の画像データを読み込みなさい。

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('chapel.jpg')

読み込んだデータはBGR値をピクセルで羅列したnumpyのarrayデータとなっています。

<mark>練習2</mark> `chapel.jpg`を読み込んだ画像データの構造について調べなさい。

In [ ]:
img

各ピクセルの色情報は、OpenCVのフォーマットでは、BGR(青・緑・赤)の順になっていますが、そのデータをmatplotlibなどで表示しようとした場合、色情報がRGB(赤・緑・青)の順番になっているため、注意が必要です。

<mark>練習3</mark> OpenCVで`chapel.jpg`を読み込んだ画像データをmatplotlibの`imshow()`を用いて表示させなさい。

In [ ]:
import matplotlib.pyplot as plt

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) #BGRをRGBに変換
plt.imshow(img)

画像データは各ピクセルの色情報を持ったarray(行列)データとなっているため、行列の行と列の一部を取り出せば画像の一部を切り出すこともできます。

<mark>練習4</mark> `chapel.jpg`を読み込んだ画像データから、横方向に150ピクセルから450ピクセル、縦方向に550ピクセルから850ピクセル部分を取り出し、画像を表示させなさい。

In [ ]:
img_extract = img[150:450, 550:850]
plt.imshow(img_extract)

同様に、行列に対する操作として、画像の上下や左右を反転させることができます。

<mark>練習5</mark> 読み込んだ画像データの上下や左右を反転させて表示させなさい。

In [ ]:
import matplotlib.pyplot as plt

img = cv2.imread('chapel.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) #BGRをRGBに変換

plt.imshow(img[:,:,:])
plt.show()

plt.imshow(img[::-1,:,:])
plt.show()

plt.imshow(img[:,::-1,:])
plt.show()

<mark>練習6</mark> 行列に対するスライスを用いて、読み込んだ画像を間引いて変形や縮小表示を行いなさい。

In [ ]:
plt.imshow(img[::2,:,:])
plt.show()

plt.imshow(img[::,::2,:])
plt.show()

plt.imshow(img[::10,::10,:])
plt.show()

色データは各ピクセルに3次元リストとして収納されているので、各色の情報を取り出すことで、色を分解したり再合成したりすることができます。

<mark>練習7</mark> 読み込んだ画像データのから、赤色・緑色・青色のそれぞれの色データを取り出し、単色に分解した画像として表示させなさい。

In [ ]:
img_red = img[:, :, 0]
plt.imshow(img_red, cmap='Reds')
plt.show()

img_green = img[:, :, 1]
plt.imshow(img_green, cmap='Greens')
plt.show()

img_blue = img[:, :, 2]
plt.imshow(img_blue, cmap='Blues')
plt.show()

自然にある映像では色で分解しても見た目には違いがよくわからないことがあります。単に映像を表示させて見るだけでなく、これまで学んだ統計的な手法を用いて各色の情報を分析することで、その画像の特徴を捉えることができます。

<mark>練習8</mark> 読み込んだ画像データの赤色・緑色・青色それぞれのチャンネルについて、画像中に含まれる0から255の数値をヒストグラムとして表示させなさい。

In [ ]:
img_red = img[:, :, 0]
plt.hist(img_red.flatten(), bins=256, color='red', alpha=0.5)

img_green = img[:, :, 1]
plt.hist(img_green.flatten(), bins=256, color='green', alpha=0.5)

img_blue = img[:, :, 2]
plt.hist(img_blue.flatten(), bins=256, color='blue', alpha=0.5)

plt.show()

### フィルタによる画像処理

画像の一つのピクセルに注目し、その周りのピクセルの情報を用いて新しい画像データを生成して処理する方法をフィルタと言います。フィルタには単純な和や差を用いただけのものや、複雑な数学演算を用いたものなど多くの種類があり、画像認識や機械学習などで重要な働きをします。

例として、$3\times 3$のピクセル領域を用いて、フィルタ処理を行うことを考えます。いま注目している(中心にある)ピクセルの座標を$(x,y)$とすれば、周りのピクセルの座標は以下のようになります。
$$\begin{array}{|c|c|c|}
\hline
(x-1,y-1)&(x,y-1)&(x+1,y-1)\\
\hline
(x-1,y)&(x,y)&(x+1,y)\\
\hline
(x-1,y+1)&(x,y+1)&(x+1,y+1)\\
\hline
\end{array}$$

ピクセル座標$(x,y)$での値(例えば色データ)を$f(x,y)$とすると、新しく更新すべき値$f'(x,y)$は周りのピクセルから以下のように計算されます。
$$\begin{split}f'(x,y)&=K(-1,-1)f(x-1,y-1)\\
&\quad+K(0,-1)f(x,y-1)\\
&\quad+K(1,-1)f(x+1,y-1)\\
&\quad+K(-1,0)f(x-1,y)\\
&\quad+K(0,0)f(x,y)\\
&\quad+K(1,0)f(x+1,y)\\
&\quad+K(-1,1)f(x-1,y+1)\\
&\quad+K(0,1)f(x,y+1)\\
&\quad+K(1,1)f(x+1,y+1)\\
&=\sum_{i,j=-1,0,1} K(i,j)f(x+i,y+j)
\end{split}$$
ここで、$K(i,j)$ ($i,j=-1,0,1$)はフィルタの係数を表す行列(カーネルという)です。

画像データを分析する手法として、ピクセルの値(輝度)について微分(差分)演算を施す方法があります。Sobelフィルタ（ピクセルデータの微分）

例えば、横方向に対して、単純な差分演算を行うカーネルは、
$$
K = \begin{pmatrix}
0 & 0 & 0 \\
-\frac{1}{2} & 0 & \frac{1}{2} \\
0 & 0 & 0 
\end{pmatrix}
$$
で、
$$
\begin{split}
f'(x,y) &= \frac{f(x+1,y) - f(x-1,y)}{2}\\
&\equiv \nabla_x f(x,y)
\end{split}
$$
となります。

<mark>練習9</mark> 白黒2値だけで図形が描かれた画像ファイル`figures.png`に対して$x$方向の差分を演算(フィルタを適用)して表示させなさい。

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

img = cv2.imread('figures.png', 0) # 0はグレースケールで読み込むオプション

y_size, x_size = img.shape
out = np.zeros((y_size, x_size), dtype=np.uint8)

# そのままだと計算が整数型で行われるため、float64に変換
img = np.float64(img)

# 差分フィルタの適用(xとyの順番に注意)
for y in range(1, y_size-1):
    for x in range(1, x_size-1):
        out[y,x] =  img[y,x+1]/2 - img[y,x-1]/2

plt.imshow(out, cmap='gray')

このように差分演算を適用すると、輝度変化の大きい部分が強調され、エッジが強調されます。これをエッジ検出と呼びます。エッジ検出は画像処理やコンピュータビジョンの分野で非常に重要な技術です。エッジ検出を行うことで、物体の輪郭や形状を抽出することができます。

ただし、一般の画像ではコントラスト変化の少ない部分も存在するため、その部分も含めてエッジを強調するために、重みづけを変化させながら周辺の差分も合わせたSobelフィルタ
$$
K_{\rm Sobel} = \frac{1}{8}\begin{pmatrix}
-1 & 0 & 1 \\
-2 & 0 & 2 \\
-1 & 0 & 1
\end{pmatrix}
$$
を適用させます。

<mark>練習10</mark> 白黒2値だけで図形が描かれた画像ファイル`figures.png`に対して$x$方向のSobelフィルタを適用して表示させなさい。

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

img = cv2.imread('figures.png', 0) # 0はグレースケールで読み込むオプション

y_size, x_size = img.shape
out = np.zeros((y_size, x_size), dtype=np.uint8)

# そのままだと計算が整数型で行われるため、float64に変換
img = np.float64(img)

# Sobelフィルタの適用(xとyの順番に注意)
for y in range(1, y_size-1):
    for x in range(1, x_size-1):
        out[y,x] = img[y-1,x+1]/8 - img[y-1,x-1]/8 + img[y,x+1]/4 -img[y,x-1]/4 + img[y+1,x+1]/8 - img[y+1,x-1]/8


plt.imshow(out, cmap='gray')

これまでの例では$x$方向の差分のみを考えてきましたが、同様に$y$方向の差分で$y$方向のエッジを検出することができます。

$x$や$y$方向だけでなく、様々な方向を含めたエッジの情報を検出するためには、勾配(gradient)
$$
\vec{\nabla}f(x,y) \equiv (\nabla_x f(x,y), \nabla_y f(x,y))
$$
を計算します。

勾配はベクトルなので、各地点で大きさと方向を持った量です。ベクトルを表現する方法は様々ですが、2次元ベクトルを2次元の画像情報として可視化する方法として、ベクトルの大きさをピクセルの明るさ、ベクトルの方向を色(色相)として表示させる方法があります。

📝 2次元ベクトル$\vec{v}=(a,b)$に対して、ベクトルの大きさは$\sqrt{a^2+b^2}$、ベクトルの方向(角度)は$\tan^{-1}\left(\frac{b}{a}\right)$で与えられます。

色空間の情報の与え方として、RGB以外に、HSVといった色相(Hue)、彩度(Saturation)、明度(Value)の3つの成分で色を表現する方法があります。

![HSV](figs/hsv.webp)

<mark>練習10</mark> 白黒2値だけで図形が描かれた画像ファイル`figures.png`に対して、グレースケールの勾配を計算し、勾配の大きさを明度(V)、勾配の方向を色相(H)として表示させなさい。(明度(S)については255の値に固定する。)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def Hue(theta):
    if theta < 0:
        theta += np.pi * 2
    return int(theta / (np.pi * 2) * 180)

img = cv2.imread('figures.png', 0)

y_size, x_size = img.shape
out = np.zeros((y_size, x_size, 3), dtype=np.uint8) # HSV画像を格納するための配列を用意

# そのままだと計算が整数型で行われるため、float64に変換
img = np.float64(img)

# 差分の計算
for y in range(1, y_size-1):
    for x in range(1, x_size-1):
        d_x =  img[y,x+1]/2 - img[y,x-1]/2
        d_y =  img[y+1,x]/2 - img[y-1,x]/2
        
        V = np.sqrt(2.0*(d_x**2 + d_y**2)) # ベクトルの大きさを計算
        theta = np.arctan2(-d_y, -d_x) # ベクトルの角度を計算 (theta=-π~π)
        H = Hue(theta) # 角度を0-180の値に変換
        S = 255

        out[y,x] = np.array([H, S, V]) # HSV画像に格納

hsv = cv2.cvtColor(out, cv2.COLOR_HSV2RGB) # HSVからRGBに変換

plt.imshow(hsv)

### Canny法

次のようなエッジを検出するアルゴリズムをCanny法(の一部)といいます。
- 輪郭線を抽出するためにソーベルフィルタをかける
- 輝度の勾配の方向と大きさを計算
- 細線化を行うために「非極大値抑制処理」をする
- 誤検知したエッジを除去するために 「しきい値処理」をする

以下でCanny法による画像処理を行います。

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def Hue(theta):
    if theta < 0:
        theta += np.pi * 2
    return int(theta / (np.pi * 2) * 180)

img = cv2.imread('figures.png', 0)

y_size, x_size = img.shape
out = np.zeros((y_size, x_size, 2)) # 勾配の大きさと角度を格納するための配列を用意

# そのままだと計算が整数型で行われるため、float64に変換
img = np.float64(img)

# 勾配の計算
for y in range(1, y_size-1):
    for x in range(1, x_size-1):
        d_x =  img[y,x+1]/2.0 - img[y,x-1]/2.0
        d_y =  img[y+1,x]/2.0 - img[y-1,x]/2.0
        
        V = np.sqrt(2.0*(d_x**2 + d_y**2)) # ベクトルの大きさを計算
        theta = np.arctan2(-d_x, -d_y) # ベクトルの角度を計算 (theta=-π~π)

        out[y,x] = np.array([V, theta]) # 配列に格納

# Canny法によるエッジの抽出
edge = np.zeros((y_size, x_size), dtype=np.uint8) # エッジを格納するための配列を用意
th = 100 # 閾値を設定

for y in range(1, y_size-1):
    for x in range(1, x_size-1):
        # Hueの値を元の角度に戻す
        t = out[y,x,1]
        dt = np.pi / 8
        
        if (-dt <= t < dt) or (7*dt <= t) or (t < -7*dt):
            if (out[y,x,0] >= out[y,x-1,0]) and (out[y,x,0] >= out[y,x+1,0]) and (out[y,x,0] > th):
                edge[y,x] = 255
        elif (dt <= t < 3*dt) or (-7*dt <= t < -5*dt):
            if (out[y,x,0] >= out[y+1,x+1,0]) and (out[y,x,0] >= out[y-1,x-1,0])  and (out[y,x,0] > th):
                edge[y,x] = 255
        elif (3*dt <= t < 5*dt) or (-5*dt <= t < -3*dt):
            if (out[y,x,0] >= out[y-1,x,0]) and (out[y,x,0] >= out[y+1,x,0]) and (out[y,x,0] > th):
                edge[y,x] = 255
        elif (5*dt <= t < 7*dt) or (-3*dt <= t < -dt):
            if (out[y,x,0] >= out[y+1,x-1,0]) and (out[y,x,0] >= out[y-1,x+1,0])  and (out[y,x,0] > th):
                edge[y,x] = 255
        
plt.imshow(edge, cmap='gray')

苦労してCanny法を実装しなくても、OpenCVにはもともとCanny法の関数が実装されています。

In [ ]:
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('figures.png', 0)
edge = cv2.Canny(img, 100, 200)

plt.imshow(edge, cmap='gray')